# 9.2 解码策略优化本节涵盖：投机解码, Medusa, Eagle

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fimport timetorch.manual_seed(42)class DraftModel(nn.Module):    def __init__(self, vocab=500, d=64):        super().__init__()        self.net = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Linear(128, vocab))    def forward(self, x):        return self.net(x)class TargetModel(nn.Module):    def __init__(self, vocab=500, d=64):        super().__init__()        self.net = nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, vocab))    def forward(self, x):        return self.net(x)draft = DraftModel()target = TargetModel()x = torch.randn(1, 64)print('=== Speculative Decoding ===')print()# Draft model generates K tokensK = 5draft_tokens = []with torch.no_grad():    for _ in range(K):        logits = draft(x)        token = logits.argmax(dim=-1)        draft_tokens.append(token)# Target model verifies all K tokens in one forward passwith torch.no_grad():    target_logits = target(x)accept_rate = 0.7  # typical acceptance ratespeedup = (1 + K * accept_rate) / (1 + 1)  # simplifiedprint(f'Draft model generates {K} candidate tokens')print(f'Target model verifies all {K} tokens in ONE forward pass')print(f'Typical acceptance rate: ~{accept_rate:.0%}')print(f'Effective speedup: ~{speedup:.1f}x')print()print('=== Medusa ===')print('Add multiple prediction heads to the model')print('Each head predicts a different future position')print('No separate draft model needed')print('Medusa-1: Single forward pass, multiple heads')print('Medusa-2: Fine-tune heads for better accuracy')print()print('=== Eagle ===')print('Use features (not just logits) from the target model for drafting')print('Features contain richer information -> higher acceptance rate')print('Typical acceptance: 80-90% (vs 60-70% for standard speculative decoding)')print('2-3x speedup on common LLMs')